## nb42 — RQ2 Subgroup: Junior vs Senior Awardees

Does the career boost from winning a best paper award differ by career stage?

- **Junior** (`career_age_at_award ≤ 5`): early-career researchers
- **Senior** (`career_age_at_award > 5`): established researchers

Two outcome lenses:
1. **Citation Lift** — from `data/matched/junior_authors_all_conferences.csv` + `pairs_lift_clean.csv`
2. **CD5 Trajectories** — from `data/cd_trajectory/` panel

All data reused from existing pipeline — no new API calls needed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.formula.api as smf
from scipy import stats
from pathlib import Path

ROOT     = Path('..')
MATCHED  = ROOT / 'data' / 'matched'
CD_DIR   = ROOT / 'data' / 'cd_trajectory'
FIG_DIR  = ROOT / 'data' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

### 1. Build seniority labels

In [ ]:
# junior_authors_all_conferences has career_age_at_award for every awardee
junior_all = pd.read_csv(MATCHED / 'junior_authors_all_conferences.csv')
print(junior_all.columns.tolist())
print(junior_all.shape)
print(junior_all.head(3))

In [ ]:
# Build seniority map: author_id -> is_junior
# Junior = career_age_at_award <= 5
seniority = junior_all[['author_id', 'career_age_at_award']].drop_duplicates('author_id').copy()
seniority['is_junior'] = (seniority['career_age_at_award'] <= 5).astype(int)

print('Junior awardees :', seniority['is_junior'].sum())
print('Senior awardees :', (seniority['is_junior'] == 0).sum())
print(seniority['career_age_at_award'].describe())

### 2. Citation Lift — Junior vs Senior

In [ ]:
pairs_lift = pd.read_csv(MATCHED / 'pairs_lift_clean.csv')
print(pairs_lift.columns.tolist())
print(pairs_lift.shape)
print(pairs_lift.head(3))

In [ ]:
# Merge seniority onto lift data (awardee side)
lift = pairs_lift.merge(seniority[['author_id', 'is_junior']], on='author_id', how='inner')
print('After seniority merge:', lift.shape)
print(lift.groupby('is_junior')['author_id'].nunique().rename({0: 'senior', 1: 'junior'}))

In [ ]:
# Descriptive stats: mean lift by seniority
lift_stats = lift.groupby('is_junior')['lift'].agg(['mean', 'median', 'std', 'count'])
lift_stats.index = lift_stats.index.map({0: 'Senior (>5 yrs)', 1: 'Junior (≤5 yrs)'})
print(lift_stats)

In [ ]:
# Mann-Whitney U test: does lift differ between junior and senior?
junior_lift  = lift.loc[lift['is_junior'] == 1, 'lift'].dropna()
senior_lift  = lift.loc[lift['is_junior'] == 0, 'lift'].dropna()

u_stat, p_val = stats.mannwhitneyu(junior_lift, senior_lift, alternative='two-sided')
print(f'Mann-Whitney U = {u_stat:.0f},  p = {p_val:.4f}')

# t-test as well
t_stat, t_p = stats.ttest_ind(junior_lift, senior_lift)
print(f'Welch t = {t_stat:.3f},  p = {t_p:.4f}')

In [ ]:
# Boxplot: lift distribution by seniority
fig, ax = plt.subplots(figsize=(7, 5))

groups = [senior_lift.values, junior_lift.values]
labels = ['Senior (>5 yrs)', 'Junior (≤5 yrs)']
bp = ax.boxplot(groups, labels=labels, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))

colors = ['#4393c3', '#d6604d']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_ylabel('Citation Lift (award − control)', fontsize=12)
ax.set_title('Citation Lift: Junior vs Senior Awardees', fontsize=13)
ax.text(0.99, 0.01, f'MW p = {p_val:.3f}', transform=ax.transAxes,
        ha='right', va='bottom', fontsize=9, color='dimgrey')

plt.tight_layout()
plt.savefig(FIG_DIR / 'nb42_lift_junior_vs_senior.png', dpi=150)
plt.show()

### 3. CD5 Trajectories — Junior vs Senior

In [ ]:
panel = pd.read_csv(CD_DIR / 'author_papers_panel.csv')
cd5   = pd.read_csv(CD_DIR / 'paper_cd5_scores.csv')

print('Panel columns:', panel.columns.tolist())
print('CD5 columns  :', cd5.columns.tolist())

In [ ]:
panel['is_awardee'] = (panel['group'] == 'award').astype(int)

# Merge CD5
merged = panel.merge(cd5[['paper_id', 'cd5']], on='paper_id', how='inner')

# Merge seniority — only awardees have a seniority label;
# for controls we can inherit the label from their matched awardee
# For now we tag awardees directly and drop unmatched controls
merged = merged.merge(seniority[['author_id', 'is_junior']], on='author_id', how='left')

# For control authors, look up their matched awardee's seniority
pairs = pd.read_csv(MATCHED / 'matched_pairs_clean.csv')
print('Matched pairs cols:', pairs.columns.tolist())
print(pairs.head(3))

In [ ]:
# Build control_id -> awardee seniority map
# pairs has awardee_id / control_id (adjust col names if needed after print above)
award_col   = [c for c in pairs.columns if 'award' in c.lower()][0]
control_col = [c for c in pairs.columns if 'control' in c.lower()][0]
print(f'Using: award_col={award_col}, control_col={control_col}')

ctrl_sen = (pairs[[award_col, control_col]]
            .merge(seniority[['author_id', 'is_junior']],
                   left_on=award_col, right_on='author_id', how='inner')
            .rename(columns={control_col: 'author_id'})[['author_id', 'is_junior']]
            .drop_duplicates('author_id'))

# Fill in missing is_junior for control authors
idx_missing = merged['is_junior'].isna()
fill = merged.loc[idx_missing, 'author_id'].map(ctrl_sen.set_index('author_id')['is_junior'])
merged.loc[idx_missing, 'is_junior'] = fill.values

merged = merged.dropna(subset=['is_junior'])
merged['is_junior'] = merged['is_junior'].astype(int)
print('Merged shape:', merged.shape)
print(merged.groupby(['is_awardee', 'is_junior'])['author_id'].nunique())

In [ ]:
# Keep ±5 window
merged = merged[merged['relative_year'].between(-5, 5)]

# Author-year aggregation
author_year = (
    merged
    .groupby(['author_id', 'is_awardee', 'is_junior', 'award_year', 'relative_year', 'paper_year'])
    ['cd5'].mean()
    .reset_index()
    .rename(columns={'cd5': 'mean_cd5', 'paper_year': 'cal_year'})
)
print('Author-year panel:', author_year.shape)

In [ ]:
def ci95(x):
    n = len(x)
    if n < 2: return np.nan
    return 1.96 * stats.sem(x, nan_policy='omit')

traj = (
    author_year
    .groupby(['is_junior', 'is_awardee', 'relative_year'])['mean_cd5']
    .agg(mean='mean', ci=ci95, n='count')
    .reset_index()
)
print(traj)

In [ ]:
# Plot CD5 trajectories side-by-side: junior (left) and senior (right)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

group_labels = {0: 'Senior (>5 yrs)', 1: 'Junior (≤5 yrs)'}
line_styles  = {1: dict(color='#d6604d', marker='o', label='Awardee'),
                0: dict(color='#4393c3', marker='s', label='Control')}

for ax, is_junior in zip(axes, [1, 0]):
    sub = traj[traj['is_junior'] == is_junior]
    for is_aw, style in line_styles.items():
        row = sub[sub['is_awardee'] == is_aw].sort_values('relative_year')
        ax.errorbar(row['relative_year'], row['mean'],
                    yerr=row['ci'], capsize=3,
                    linewidth=2, markersize=5, **style)
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
    ax.axhline(0, color='lightgrey', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Years relative to award', fontsize=11)
    ax.set_title(group_labels[is_junior], fontsize=12)
    ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
    ax.legend(fontsize=9)

axes[0].set_ylabel('Mean CD5 score', fontsize=11)
fig.suptitle('CD5 Trajectories: Junior vs Senior Awardees (±5 years)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'nb42_cd5_junior_vs_senior.png', dpi=150, bbox_inches='tight')
plt.show()

### 4. TWFE DiD — interaction is_awardee × is_junior × rel_year

In [ ]:
# Drop reference year (-1) for clean identification
ay_did = author_year[author_year['relative_year'] != -1].copy()

# TWFE with triple interaction
# FE: author_id (unit) + cal_year (time)
formula = ('mean_cd5 ~ C(relative_year) * is_awardee * is_junior'
           ' + C(author_id) + C(cal_year)')

model = smf.ols(formula, data=ay_did).fit(cov_type='HC3')
print(model.summary2().tables[1][    model.summary2().tables[1].index.str.contains('is_awardee|is_junior')
])

In [ ]:
# Simpler: separate DiD per subgroup to get ATT estimates
results = {}
for label, is_jn in [('Junior (≤5 yrs)', 1), ('Senior (>5 yrs)', 0)]:
    sub = ay_did[ay_did['is_junior'] == is_jn].copy()
    m = smf.ols('mean_cd5 ~ C(relative_year) * is_awardee + C(author_id) + C(cal_year)',
                data=sub).fit(cov_type='HC3')
    # post-award ATT = mean of interaction terms for rel_year > 0
    params = m.params
    post_terms = [p for p in params.index
                  if 'is_awardee' in p and 'relative_year' in p
                  and int(p.split('[T.')[1].split(']')[0]) > 0]
    att = params[post_terms].mean()
    pvals = m.pvalues[post_terms]
    results[label] = {'ATT_post': att, 'min_p': pvals.min(), 'n_sig': (pvals < 0.05).sum()}
    print(f'{label}: ATT_post = {att:.4f}, min_p = {pvals.min():.4f}, sig terms = {(pvals<0.05).sum()}/{len(post_terms)}')

print('\nDone.')

### 5. Summary table

| Subgroup | Mean Lift | CD5 ATT (post) | Min p |
|---|---|---|---|
| Junior (≤5 yrs) | see cell above | see cell above | see cell above |
| Senior (>5 yrs) | see cell above | see cell above | see cell above |